<!-- paper-paired-role-banner -->
> 🟦 **실습 코드 · 왼쪽 창** — TODO를 먼저 직접 구현하세요. 막히면 오른쪽 정답의 같은 셀에서 필요한 한 줄만 확인합니다.

# 09. Attention Is All You Need — Attention Is All You Need

- **원 논문:** [Attention Is All You Need](https://arxiv.org/abs/1706.03762)
- **저자 / 발표:** Ashish Vaswani et al. · NeurIPS 2017 (2017)
- **난이도 / 예상 시간:** 중상급 · 약 80분
- **선수 지식:** 선형대수, softmax, broadcasting, attention mask

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

식 (1)의 scaled dot-product attention, 식 (2)--(3)의 multi-head projection, §3.3의 position-wise FFN, §3.1의 residual connection과 post-LayerNorm, §3.5의 sinusoidal positional encoding을 직접 구현한다. 같은 block을 encoder처럼 mask 없이, decoder self-attention처럼 causal mask와 함께 실행해 미래 token 불변성, 확률 정규화, scale이 entropy에 미치는 효과를 검증한다.

**원 논문과 다른 것**

원 논문의 base Transformer는 6층 encoder/decoder, d_model=512, 8 heads로 WMT'14 En-De 약 450만 문장 쌍을 학습했다. 이 노트북은 d_model=16, 4 heads인 단일 self-attention+FFN block을 사용한다. encoder-decoder cross-attention, 6층 stack, label smoothing, 번역 학습과 beam search는 재현하지 않는다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §3.2.1, 식 (1): Attention(Q,K,V)=softmax(QKᵀ/√d_k)V | `scaled_dot_product_attention`의 score→mask→softmax→weighted sum | attention 행 합, shape, 유한값을 assert |
| §3.2.2, 식 (2)--(3): head별 projection 후 concat과 W^O | `MultiHeadSelfAttention`의 projection/split/transpose/merge | 출력과 head별 weight shape 검사 |
| §3.1: decoder self-attention에서 subsequent position 차단 | `causal_attention_mask`와 `TransformerBlock(..., allowed=causal)` | 미래 attention mass=0 및 prefix 출력 불변성 assert |
| §3.5: sinusoidal positional encoding 수식 | `sinusoidal_positions`의 짝수 sin/홀수 cos 구현 | position 0 패턴과 shape 검사 |
| §3.2.1: 1/√d_k scaling이 큰 내적으로 인한 작은 gradient를 완화 | `attention_entropy`의 scaled/unscaled score 비교 | scaled entropy가 더 큰지 정량 비교하고 막대그래프로 표시 |
| §3.1: Sublayer 출력은 LayerNorm(x + Sublayer(x)) | `TransformerBlock`의 attention/FFN residual 뒤 각각 post-LayerNorm | residual shape와 token별 평균·분산을 검사 |
| §3.3, 식 FFN(x)=max(0,xW1+b1)W2+b2 | `PositionWiseFeedForward`의 두 Linear와 ReLU | 동일 module이 모든 token 위치에 독립 적용되는 shape를 검사 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
\operatorname{Attention}(Q,K,V)=
\operatorname{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}+M\right)V
$$

- $Q,K,V$는 query·key·value, $d_k$는 head dimension, $M$은 causal/padding mask입니다.
- $1/\sqrt{d_k}$ scaling은 dot-product variance를 줄여 softmax saturation을 방지합니다.
- projection, head split/merge, mask, attention weight를 순서대로 직접 구현합니다.
- $[B,T,D]\rightarrow[B,H,T,d_k]\rightarrow[B,T,D]$의 shape 변화를 assertion으로 확인합니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §3.2.1, 식 (1): Attention(Q,K,V)=softmax(QKᵀ/√d_k)V
- **노트북 구현:** `scaled_dot_product_attention`의 score→mask→softmax→weighted sum
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** attention 행 합, shape, 유한값을 assert를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** 원 논문의 base Transformer는 6층 encoder/decoder, d_model=512, 8 heads로 WMT'14 En-De 약 450만 문장 쌍을 학습했다. 이 노트북은 d_model=16, 4 heads인 단일 self-attention+FFN block을 사용한다. encoder-decoder cross-attention, 6층 stack, label smoothing, 번역 학습과 beam search는 재현하지 않는다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 핵심 아이디어

Transformer는 recurrence 없이 attention만으로 위치 간 정보를 섞는다.
dot product의 분산은 차원 $d_k$와 함께 커지므로 식 (1)은 점수를
$\sqrt{d_k}$로 나눈다. 여러 projection 공간에서 병렬로 attention한 결과를
이어 붙이는 것이 multi-head attention이다. 순서는 고정 sin/cos 위치 벡터를
token 표현에 더해 제공하며, decoder에는 미래를 볼 수 없는 causal mask가 필요하다.

§3.1의 원 논문 block은 각 sublayer 뒤에
$\mathrm{LayerNorm}(x+\mathrm{Sublayer}(x))$를 적용하는 **post-LN** 구조다.
여기서도 이 순서를 그대로 드러내며, §3.3의 position-wise FFN도 고수준
Transformer module 안에 숨기지 않는다. `allowed=None`은 encoder self-attention,
lower-triangular mask는 decoder의 첫 masked self-attention sublayer에 대응한다.
단, decoder의 encoder-decoder attention은 이 축소 실험 범위 밖이다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** hidden [B,T,D] ↔ heads [B,H,T,dₖ] → block output [B,T,D]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
import math

import matplotlib.pyplot as plt
import torch
from torch import nn
from llm_engineering_lab.acceleration import get_accelerator

SEED = 909
torch.manual_seed(SEED)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
print(ACCELERATOR.summary())

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §3.5: sinusoidal positional encoding 수식
- **이 셀의 책임:** `sinusoidal_positions`의 짝수 sin/홀수 cos 구현
- **Tensor shape 계약:** hidden [B,T,D] ↔ heads [B,H,T,dₖ] → block output [B,T,D]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** position 0 패턴과 shape 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 09-1: 논문 §3.5의 짝수 sin/홀수 cos positional encoding을 구현하세요.
def sinusoidal_positions(length, d_model):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


positions = sinusoidal_positions(length=6, d_model=16)


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §3.2.1, 식 (1): Attention(Q,K,V)=softmax(QKᵀ/√d_k)V
- **이 셀의 책임:** `scaled_dot_product_attention`의 score→mask→softmax→weighted sum
- **Tensor shape 계약:** hidden [B,T,D] ↔ heads [B,H,T,dₖ] → block output [B,T,D]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** attention 행 합, shape, 유한값을 assert. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 09-2: 식 (1)의 scale, mask, softmax, value 가중합을 구현하세요.
def scaled_dot_product_attention(query, key, value, allowed=None):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 4 · 핵심 연산·모델

- **원문 위치:** §3.2.1, 식 (1): Attention(Q,K,V)=softmax(QKᵀ/√d_k)V / §3.2.2, 식 (2)--(3): head별 projection 후 concat과 W^O / §3.1: decoder self-attention에서 subsequent position 차단
- **이 셀의 책임:** `scaled_dot_product_attention`의 score→mask→softmax→weighted sum / `MultiHeadSelfAttention`의 projection/split/transpose/merge / `causal_attention_mask`와 `TransformerBlock(..., allowed=causal)`
- **Tensor shape 계약:** hidden [B,T,D] ↔ heads [B,H,T,dₖ] → block output [B,T,D]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** attention 행 합, shape, 유한값을 assert / 출력과 head별 weight shape 검사 / 미래 attention mass=0 및 prefix 출력 불변성 assert. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 09-3: 식 (2)--(3)의 head split, attention, concat, W^O를 구현하세요.
def causal_attention_mask(length, device):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


class MultiHeadSelfAttention(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, d_model, num_heads):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def _split(self, tensor):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def _merge(self, tensor):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def forward(self, hidden, allowed=None):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 5 · 핵심 연산·모델

- **원문 위치:** §3.2.2, 식 (2)--(3): head별 projection 후 concat과 W^O / §3.1: decoder self-attention에서 subsequent position 차단 / §3.1: Sublayer 출력은 LayerNorm(x + Sublayer(x)) / §3.3, 식 FFN(x)=max(0,xW1+b1)W2+b2
- **이 셀의 책임:** `MultiHeadSelfAttention`의 projection/split/transpose/merge / `causal_attention_mask`와 `TransformerBlock(..., allowed=causal)` / `TransformerBlock`의 attention/FFN residual 뒤 각각 post-LayerNorm / `PositionWiseFeedForward`의 두 Linear와 ReLU
- **Tensor shape 계약:** hidden [B,T,D] ↔ heads [B,H,T,dₖ] → block output [B,T,D]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 출력과 head별 weight shape 검사 / 미래 attention mass=0 및 prefix 출력 불변성 assert / residual shape와 token별 평균·분산을 검사 / 동일 module이 모든 token 위치에 독립 적용되는 shape를 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 09-4: §3.3의 FFN과 §3.1의 post-LN residual block을 구현하세요.
class PositionWiseFeedForward(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, d_model, d_ff):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, hidden):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


class TransformerBlock(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, d_model, num_heads, d_ff):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, hidden, allowed=None):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


block = TransformerBlock(d_model=16, num_heads=4, d_ff=32).to(DEVICE)


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 6 · 평가·완료 증거

- **원문 위치:** §3.1: decoder self-attention에서 subsequent position 차단 / §3.2.1: 1/√d_k scaling이 큰 내적으로 인한 작은 gradient를 완화
- **이 셀의 책임:** `causal_attention_mask`와 `TransformerBlock(..., allowed=causal)` / `attention_entropy`의 scaled/unscaled score 비교
- **Tensor shape 계약:** hidden [B,T,D] ↔ heads [B,H,T,dₖ] → block output [B,T,D]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 미래 attention mass=0 및 prefix 출력 불변성 assert / scaled entropy가 더 큰지 정량 비교하고 막대그래프로 표시. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 09-5: full block에서도 미래 token 불변성을 검증하세요.
# TODO 09-6: scaled/unscaled score entropy를 비교하고 attention을 그리세요.
def attention_entropy(weights):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


changed_hidden = hidden.clone()
changed_hidden[:, -1] += 100.0
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

## 해석 체크

1. `sqrt(d_k)`로 나누지 않으면 softmax와 gradient에 어떤 일이 생기나요?
2. 원 논문의 post-LN 수식과 최신 pre-LN 구현의 연산 순서는 어떻게 다른가요?
3. causal mask 검증에서 마지막 token을 크게 바꾼 이유는 무엇인가요?
4. 서로 다른 head가 유용할 수 있는 이유를 projection 관점에서 설명하세요.

## 마무리 관찰 기록

아래 세 문장을 직접 완성하세요.

1. 이 논문의 핵심 연산은 `_____`이고, 코드에서는 `_____`에 해당한다.
2. 원 규모 실험 대신 미니 재현을 사용했기 때문에 확인할 수 없는 주장은 `_____`이다.
3. 한 가지 변수를 바꾸어 다시 실행한다면 `_____`를 바꾸고 `_____`를 측정하겠다.

**추가 실험:** seed는 유지한 채 한 변수만 바꾸고, 변경 전후의 metric을 표로 남기세요.